## 1. Setup
Paths and the league-code mapping between our two sources:
football-data.co.uk uses `E0/SP1/D1/I1/F1`, the football-data.org API
uses `PL/PD/BL1/SA/FL1`. We standardise on the API codes everywhere.

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np

# Notebook lives in notebooks/, data lives one level up
RAW_DIR = Path("../data/raw")

# CSV league code -> API competition code (the shared vocabulary of both sources)
LEAGUE_MAP = {
    "E0": "PL",
    "SP1": "PD",
    "D1": "BL1",
    "I1": "SA",
    "F1": "FL1",
}
CSV_SEASONS = ["2425", "2526"]

## 2. Load all odds CSVs into one frame
The ten files do **not** share a schema (119–132 columns; bookmakers
appear and disappear between seasons). `pd.concat` takes the union of
columns, so season-exclusive columns become NaN elsewhere — we measure
that in the next cell instead of letting it surprise us later.
Provenance (`league`, `season_file`) is attached via concat keys.

In [5]:
# Read each file under a (league, season) key; concat turns the keys
# into index levels, which we then promote to ordinary columns.

frames = {}
for season in CSV_SEASONS:
    for csv_code, api_code in LEAGUE_MAP.items():
        filepath = RAW_DIR / f"odds_{csv_code}_{season}.csv"
        frames[(api_code, season)] = pd.read_csv(filepath)

# .copy() defragments the frame after the union-concat,
# avoiding pandas' PerformanceWarning on subsequent inserts
odds_raw = pd.concat(frames, names=["league", "season_file"], sort=False).copy()
odds_raw = odds_raw.reset_index(level=["league", "season_file"]).reset_index(drop=True)
print(odds_raw.shape)

(3504, 152)


## 3. Missingness audit
Ranking columns by share of missing values separates two phenomena:
columns that exist only in some files (season/league schema drift,
~35–65% missing) versus genuinely sparse values. Anything we keep for
analysis must be near-complete across *all* leagues and seasons.
Findings: `Referee` (E0-only, 78%), William Hill (~62%), Coral,
Ladbrokes, 1xBet, BetVictor all too patchy to use.

In [9]:
# Share of NaN per column, worst first — the shape of the schema drift
null_share = odds_raw.isna().mean().sort_values(ascending=False)
print(null_share.head(25).round(3))

Referee    0.783
CLCD       0.650
CLCA       0.650
CLCH       0.650
CLD        0.635
CLH        0.635
CLA        0.635
LBD        0.626
LBH        0.626
LBA        0.626
LBCD       0.626
LBCH       0.626
LBCA       0.626
WHCA       0.622
WHCH       0.622
WHCD       0.622
WHD        0.622
WHA        0.622
WHH        0.622
1XBA       0.513
1XBD       0.513
1XBH       0.513
BVCH       0.511
BVCA       0.511
BVCD       0.511
dtype: float64


In [7]:
# Targeted completeness check on keep-list candidates.
# Rule: a bookmaker only stays in the analysis if its closing 1X2 odds
# are near-complete across ALL leagues and both seasons — otherwise
# cross-league margin comparisons would silently compare different samples.
keep_candidates = [
    "B365CH", "B365CD", "B365CA",   # Bet365 closing 1X2
    "PSCH", "PSCD", "PSCA",         # Pinnacle closing 1X2
    "BWCH", "BWCD", "BWCA",         # Bwin closing 1X2
    "MaxCH", "MaxCD", "MaxCA",      # market maximum, closing
    "AvgCH", "AvgCD", "AvgCA",      # market average, closing
]
print(odds_raw[keep_candidates].isna().mean().round(4))

B365CH    0.0000
B365CD    0.0000
B365CA    0.0000
PSCH      0.2437
PSCD      0.2437
PSCA      0.2437
BWCH      0.1909
BWCD      0.1909
BWCA      0.1909
MaxCH     0.0000
MaxCD     0.0000
MaxCA     0.0000
AvgCH     0.0000
AvgCD     0.0000
AvgCA     0.0000
dtype: float64


In [10]:
# Where exactly are Pinnacle and Bwin missing? Random scatter is tolerable;
# concentration by league/season means the column can't support cross-league work.
missing_structure = (
    odds_raw
    .groupby(["league", "season_file"])[["PSCH", "BWCH"]]
    .apply(lambda g: g.isna().mean())
    .round(3)
)
print(missing_structure)

                     PSCH   BWCH
league season_file              
BL1    2425         0.000  0.382
       2526         0.513  0.000
FL1    2425         0.000  0.382
       2526         0.500  0.000
PD     2425         0.000  0.397
       2526         0.505  0.003
PL     2425         0.000  0.371
       2526         0.447  0.000
SA     2425         0.000  0.374
       2526         0.479  0.000


In [11]:
# Is the missingness early-season or late-season? This decides whether our
# calendar-2025 window is affected at all.
tmp = odds_raw.copy()
tmp["date_parsed"] = pd.to_datetime(tmp["Date"], dayfirst=True, format="mixed")
tmp["in_2025"] = tmp["date_parsed"].dt.year == 2025

coverage = (
    tmp.groupby(["season_file", "in_2025"])[["PSCH", "BWCH"]]
    .apply(lambda g: g.isna().mean())
    .round(3)
)
print(coverage)
print("\nRows in calendar 2025:", tmp["in_2025"].sum())

                      PSCH   BWCH
season_file in_2025              
2425        False    0.000  0.000
            True     0.000  0.715
2526        False    0.898  0.001
            True     0.001  0.000

Rows in calendar 2025: 1736


## 4. Keep-list and tidy odds table
Decision trail: WH/Coral/Ladbrokes/1xBet/BetVictor dropped (coverage),
Bwin dropped (71.5% missing Jan–May 2025, inside the analysis window),
Pinnacle retained (its gap is Jan–May 2026, outside the window).
Final: Bet365 + Pinnacle closing 1X2 + market Max/Avg + match core,
filtered to calendar-2025 kickoffs.

In [12]:
KEEP_COLS = {
    # match core
    "league": "league",
    "season_file": "season_file",
    "Date": "date",
    "Time": "kickoff_local",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "FTR": "result",
    # closing 1X2 odds
    "B365CH": "b365_home", "B365CD": "b365_draw", "B365CA": "b365_away",
    "PSCH": "pin_home",   "PSCD": "pin_draw",   "PSCA": "pin_away",
    "MaxCH": "max_home",  "MaxCD": "max_draw",  "MaxCA": "max_away",
    "AvgCH": "avg_home",  "AvgCD": "avg_draw",  "AvgCA": "avg_away",
}

odds = odds_raw[list(KEEP_COLS)].rename(columns=KEEP_COLS).copy()
odds["date"] = pd.to_datetime(odds["date"], dayfirst=True, format="mixed")
odds = odds[odds["date"].dt.year == 2025].reset_index(drop=True)

print(odds.shape)                 # expect (1736, 21)
print(odds["league"].value_counts())
odds.head()

(1736, 21)
league
PL     378
PD     370
SA     368
FL1    314
BL1    306
Name: count, dtype: int64


,league,season_file,date,kickoff_local,home_team,away_team,home_goals,away_goals,result,b365_home,...,b365_away,pin_home,pin_draw,pin_away,max_home,max_draw,max_away,avg_home,avg_draw,avg_away
0,PL,2425,2025-01-01,17:30,Brentford,Arsenal,1,3,A,5.50,...,1.55,6.00,4.50,1.56,7.16,4.90,1.57,5.87,4.46,1.54
1,PL,2425,2025-01-04,12:30,Tottenham,Newcastle,1,2,A,3.90,...,1.73,3.90,4.69,1.79,3.90,4.69,1.94,3.81,4.50,1.79
2,PL,2425,2025-01-04,15:00,Aston Villa,Leicester,2,1,H,1.33,...,8.50,1.34,5.69,8.32,1.37,6.00,9.50,1.33,5.63,8.49
3,PL,2425,2025-01-04,15:00,Bournemouth,Everton,1,0,H,1.67,...,5.00,1.70,3.97,5.23,1.72,4.00,5.25,1.69,3.86,5.10
4,PL,2425,2025-01-04,15:00,Crystal Palace,Chelsea,1,1,D,3.60,...,1.91,3.64,3.94,2.00,3.89,4.00,2.05,3.58,3.84,1.98
